# Olist E-Commerce: Customer Retention Analysis
## 01 - Data Exploration

**Objective:** Understand the 9 Olist datasets and identify the critical business questions around customer retention.

---

### 🔑 Key Findings (Day 1)

**1. Dataset overview:**
- 9 related tables covering orders (2016-2018), customers, products, sellers, payments, reviews, geolocation
- ~99K orders, ~96K unique customers, ~33K products

**2. Critical insight — `customer_id` is misleading:**
- `customer_id` = one per order (not per person)
- `customer_unique_id` = the actual person
- Using `customer_unique_id` reveals the true retention picture

**3. Olist has a major retention problem:**
- **Repeat purchase rate: 3.12%**
- 96.88% of customers never return
- Industry benchmark: 20-40%

**4. Top repeat customer paradox:**
- Most frequent customer: 17 orders, all from São Paulo
- BUT their avg order value (R$ 53) is 3x LOWER than overall average (R$ 159)
- Implication: frequency ≠ value at Olist; retention strategy must segment both

**5. Data quality issues to fix:**
- All date columns stored as text (need datetime conversion)
- 3% of orders have no delivery date (likely cancelled)
- Product categories in Portuguese (need English translation join)
- 1.9% products missing category

---

### Questions to investigate next
1. Do delivery delays cause customers not to return?
2. Does review score predict repeat purchase?
3. Which product categories have the highest retention?
4. What's the typical time-to-second-purchase for repeat customers?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sns.set_style('whitegrid')

DATA_PATH = '../data/raw/'

print("Setup complete! Libraries loaded.")

Setup complete! Libraries loaded.


In [2]:
# Load all 9 datasets into a dictionary
datasets = {
    'customers': pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv'),
    'geolocation': pd.read_csv(DATA_PATH + 'olist_geolocation_dataset.csv'),
    'order_items': pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv'),
    'order_payments': pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv'),
    'order_reviews': pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv'),
    'orders': pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv'),
    'products': pd.read_csv(DATA_PATH + 'olist_products_dataset.csv'),
    'sellers': pd.read_csv(DATA_PATH + 'olist_sellers_dataset.csv'),
    'category_translation': pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')
}

# Print the shape (rows, columns) of each dataset
print("Dataset sizes:")
print("-" * 50)
for name, df in datasets.items():
    print(f"{name:25s} → {df.shape[0]:>8,} rows × {df.shape[1]} columns")

Dataset sizes:
--------------------------------------------------
customers                 →   99,441 rows × 5 columns
geolocation               → 1,000,163 rows × 5 columns
order_items               →  112,650 rows × 7 columns
order_payments            →  103,886 rows × 5 columns
order_reviews             →   99,224 rows × 7 columns
orders                    →   99,441 rows × 8 columns
products                  →   32,951 rows × 9 columns
sellers                   →    3,095 rows × 4 columns
category_translation      →       71 rows × 2 columns


In [3]:
# Show a preview of each dataset — columns and first 3 rows
for name, df in datasets.items():
    print("=" * 80)
    print(f"📊 {name.upper()}")
    print("=" * 80)
    print(f"Columns: {list(df.columns)}")
    print(f"\nFirst 3 rows:")
    print(df.head(3))
    print("\n")

📊 CUSTOMERS
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

First 3 rows:
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  


📊 GEOLOCATION
Columns: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

First 3 rows:
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1037       -23.545621       -46.639292   
1          

In [4]:
# Check for missing values, duplicates, and data types
for name, df in datasets.items():
    print("=" * 70)
    print(f"🔍 {name.upper()}")
    print("=" * 70)
    
    # Missing values
    missing = df.isnull().sum()
    missing = missing[missing > 0]  # only show columns with missing values
    if len(missing) > 0:
        print("\n❗ Missing values:")
        for col, count in missing.items():
            pct = (count / len(df)) * 100
            print(f"   {col:40s} {count:>8,} ({pct:.1f}%)")
    else:
        print("\n✅ No missing values")
    
    # Duplicates
    dupes = df.duplicated().sum()
    print(f"\n🔁 Duplicate rows: {dupes:,}")
    
    # Data types
    print(f"\n📋 Data types:")
    print(df.dtypes.to_string())
    print()

🔍 CUSTOMERS

✅ No missing values

🔁 Duplicate rows: 0

📋 Data types:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object

🔍 GEOLOCATION

✅ No missing values

🔁 Duplicate rows: 261,831

📋 Data types:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object

🔍 ORDER_ITEMS

✅ No missing values

🔁 Duplicate rows: 0

📋 Data types:
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64

🔍 ORDER_PAYMENTS

✅ No missing values

🔁 Duplicate rows: 0

📋 Data types:
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments   

In [5]:
customers = datasets['customers']

total_rows = len(customers)
unique_customer_ids = customers['customer_id'].nunique()
unique_customer_unique_ids = customers['customer_unique_id'].nunique()

print(f"Total rows in customers: {total_rows:,}")
print(f"Unique customer_id values: {unique_customer_ids:,}")
print(f"Unique customer_unique_id values: {unique_customer_unique_ids:,}")
print(f"\nDifference: {total_rows - unique_customer_unique_ids:,} rows are 'repeat customers'")

Total rows in customers: 99,441
Unique customer_id values: 99,441
Unique customer_unique_id values: 96,096

Difference: 3,345 rows are 'repeat customers'


In [6]:
# How many orders does each actual person make?
purchase_counts = customers['customer_unique_id'].value_counts()

# Distribution of purchase counts
distribution = purchase_counts.value_counts().sort_index()

print("How many times do customers purchase?")
print("-" * 50)
for orders, count in distribution.items():
    pct = (count / len(purchase_counts)) * 100
    print(f"  {orders} order(s): {count:>6,} customers ({pct:.2f}%)")

print(f"\nTotal unique customers: {len(purchase_counts):,}")
print(f"Repeat customers (2+ orders): {(purchase_counts >= 2).sum():,}")
print(f"Repeat purchase rate: {(purchase_counts >= 2).sum() / len(purchase_counts) * 100:.2f}%")

How many times do customers purchase?
--------------------------------------------------
  1 order(s): 93,099 customers (96.88%)
  2 order(s):  2,745 customers (2.86%)
  3 order(s):    203 customers (0.21%)
  4 order(s):     30 customers (0.03%)
  5 order(s):      8 customers (0.01%)
  6 order(s):      6 customers (0.01%)
  7 order(s):      3 customers (0.00%)
  9 order(s):      1 customers (0.00%)
  17 order(s):      1 customers (0.00%)

Total unique customers: 96,096
Repeat customers (2+ orders): 2,997
Repeat purchase rate: 3.12%


In [7]:
# Find the top repeat customers
top_customers = purchase_counts.head(10)
print("Top 10 customers by order count:")
print(top_customers)
print()

# Let's look at the 17-order customer's details
top_customer_id = purchase_counts.index[0]
print(f"Top customer unique_id: {top_customer_id}\n")

# Get all their customer_ids
their_customer_ids = customers[customers['customer_unique_id'] == top_customer_id]
print(f"They have {len(their_customer_ids)} different customer_id entries")
print(f"\nTheir location(s):")
print(their_customer_ids[['customer_city', 'customer_state']].value_counts())

Top 10 customers by order count:
customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
1b6c7548a2a1f9037c1fd3ddfed95f33     7
12f5d6e1cbf93dafd9dcc19095df0b3d     6
dc813062e0fc23409cd255f7f53c7074     6
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
de34b16117594161a6a89c50b289d35a     6
63cfc61cee11cbe306bff5857d00bfe4     6
Name: count, dtype: int64

Top customer unique_id: 8d50f5eadf50201ccdcedfb9e2ac8455

They have 17 different customer_id entries

Their location(s):
customer_city  customer_state
sao paulo      SP                17
Name: count, dtype: int64


In [8]:
# Get all order_ids for the super-customer
their_customer_ids_list = customers[customers['customer_unique_id'] == top_customer_id]['customer_id'].tolist()

# Find all their orders
their_orders = datasets['orders'][datasets['orders']['customer_id'].isin(their_customer_ids_list)]
print(f"Orders placed: {len(their_orders)}")

# Get all their order items (to calculate total spend)
their_items = datasets['order_items'][datasets['order_items']['order_id'].isin(their_orders['order_id'])]
total_spend = their_items['price'].sum()
total_freight = their_items['freight_value'].sum()

print(f"Total product spend: R$ {total_spend:,.2f}")
print(f"Total freight paid: R$ {total_freight:,.2f}")
print(f"Grand total: R$ {total_spend + total_freight:,.2f}")
print(f"Average order value: R$ {(total_spend + total_freight) / len(their_orders):,.2f}")

# Compare to average customer
all_items = datasets['order_items']
all_orders = datasets['orders']
avg_order_value = (all_items['price'].sum() + all_items['freight_value'].sum()) / len(all_orders)
print(f"\nFor context, average order value across ALL customers: R$ {avg_order_value:,.2f}")

Orders placed: 17
Total product spend: R$ 729.62
Total freight paid: R$ 172.42
Grand total: R$ 902.04
Average order value: R$ 53.06

For context, average order value across ALL customers: R$ 159.33


In [9]:
# Convert date columns in ORDERS table
orders = datasets['orders'].copy()

date_cols_orders = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols_orders:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Convert date columns in ORDER_ITEMS
order_items = datasets['order_items'].copy()
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'], errors='coerce')

# Convert date columns in ORDER_REVIEWS
order_reviews = datasets['order_reviews'].copy()
order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'], errors='coerce')
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'], errors='coerce')

# Verify it worked
print("✅ Dates converted. Let's verify:\n")
print(orders[date_cols_orders].dtypes)
print(f"\nDate range of orders: {orders['order_purchase_timestamp'].min()} to {orders['order_purchase_timestamp'].max()}")

✅ Dates converted. Let's verify:

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Date range of orders: 2016-09-04 21:15:19 to 2018-10-17 17:30:18


In [10]:
# Start with orders + customers (to get the real customer_unique_id)
master = orders.merge(
    datasets['customers'], 
    on='customer_id', 
    how='left'
)
print(f"After orders + customers: {master.shape}")

# Add order items (this will create multiple rows for multi-item orders)
master = master.merge(
    order_items,
    on='order_id',
    how='left'
)
print(f"After + order_items:       {master.shape}")

# Add products
master = master.merge(
    datasets['products'][['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)
print(f"After + products:          {master.shape}")

# Translate category to English
master = master.merge(
    datasets['category_translation'],
    on='product_category_name',
    how='left'
)
print(f"After + translations:      {master.shape}")

# Add review scores (aggregate first — one review per order)
reviews_agg = order_reviews.groupby('order_id', as_index=False).agg(
    review_score=('review_score', 'mean'),
    has_review=('review_id', 'count')
)
master = master.merge(reviews_agg, on='order_id', how='left')
print(f"After + reviews:           {master.shape}")

print(f"\n✅ Master table built!")
print(f"Total rows: {master.shape[0]:,}")
print(f"Total columns: {master.shape[1]}")
print(f"\nColumns: {list(master.columns)}")


After orders + customers: (99441, 12)
After + order_items:       (113425, 18)
After + products:          (113425, 19)
After + translations:      (113425, 20)
After + reviews:           (113425, 22)

✅ Master table built!
Total rows: 113,425
Total columns: 22

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_category_name_english', 'review_score', 'has_review']


In [11]:
# Sanity checks on the master table

print("="*60)
print("SANITY CHECKS")
print("="*60)

# 1. Unique counts
print(f"\nUnique orders:      {master['order_id'].nunique():,}")
print(f"Unique customers:   {master['customer_unique_id'].nunique():,}")
print(f"Unique products:    {master['product_id'].nunique():,}")
print(f"Unique categories:  {master['product_category_name_english'].nunique():,}")

# 2. Missing values check
print(f"\nMissing values in key columns:")
key_cols = ['customer_unique_id', 'order_purchase_timestamp', 'price', 'review_score', 'product_category_name_english']
for col in key_cols:
    missing = master[col].isnull().sum()
    pct = (missing / len(master)) * 100
    print(f"  {col:40s} {missing:>6,} ({pct:.1f}%)")

# 3. Order status distribution
print(f"\nOrder status breakdown:")
print(master['order_status'].value_counts())

# 4. Revenue sanity check
total_revenue = master['price'].sum()
total_freight = master['freight_value'].sum()
print(f"\nTotal product revenue:  R$ {total_revenue:,.2f}")
print(f"Total freight revenue:  R$ {total_freight:,.2f}")
print(f"Grand total:            R$ {total_revenue + total_freight:,.2f}")


SANITY CHECKS

Unique orders:      99,441
Unique customers:   96,096
Unique products:    32,951
Unique categories:  71

Missing values in key columns:
  customer_unique_id                            0 (0.0%)
  order_purchase_timestamp                      0 (0.0%)
  price                                       775 (0.7%)
  review_score                                961 (0.8%)
  product_category_name_english             2,402 (2.1%)

Order status breakdown:
order_status
delivered      110197
shipped          1186
canceled          706
unavailable       610
invoiced          361
processing        357
created             5
approved            3
Name: count, dtype: int64

Total product revenue:  R$ 13,591,643.70
Total freight revenue:  R$ 2,251,909.54
Grand total:            R$ 15,843,553.24


In [12]:
# Are missing prices concentrated in certain order statuses?
missing_price_mask = master['price'].isnull()
print("Order status of rows with missing price:")
print(master.loc[missing_price_mask, 'order_status'].value_counts())

Order status of rows with missing price:
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


In [14]:
# Filter to delivered orders only — our analysis universe
master_delivered = master[master['order_status'] == 'delivered'].copy()

# Remove any leftover rows with missing prices (safety net)
master_delivered = master_delivered[master_delivered['price'].notna()].copy()

print(f"Original master table:  {len(master):,} rows")
print(f"Delivered + priced:     {len(master_delivered):,} rows")
print(f"Dropped:                {len(master) - len(master_delivered):,} rows")

# Create the processed folder
import os
os.makedirs('../data/processed', exist_ok=True)

# Save as parquet (much smaller and faster than CSV)
master_delivered.to_parquet('../data/processed/master_delivered.parquet', index=False)

# Also save as CSV for SQL loading later
master_delivered.to_csv('../data/processed/master_delivered.csv', index=False)

print(f"\n✅ Saved to data/processed/")
print(f"   - master_delivered.parquet (for Python)")
print(f"   - master_delivered.csv (for SQL later)")

Original master table:  113,425 rows
Delivered + priced:     110,197 rows
Dropped:                3,228 rows

✅ Saved to data/processed/
   - master_delivered.parquet (for Python)
   - master_delivered.csv (for SQL later)


In [15]:
import os
os.makedirs('../data/processed', exist_ok=True)

master_delivered.to_parquet('../data/processed/master_delivered.parquet', index=False)
master_delivered.to_csv('../data/processed/master_delivered.csv', index=False)

print(f"✅ Saved to data/processed/")
print(f"   - master_delivered.parquet (for Python)")
print(f"   - master_delivered.csv (for SQL later)")

✅ Saved to data/processed/
   - master_delivered.parquet (for Python)
   - master_delivered.csv (for SQL later)
